# Допуск: OpenHands: An Open Platform for AI Software Developers as Generalist Agents

**Статья:** X. Wang и др. — [конференционная версия ICLR 2025](https://proceedings.iclr.cc/paper_files/paper/2025/file/a4b6ad6b48850c0c331d1259fc66a69c-Paper-Conference.pdf),
[arXiv:2407.16741](https://arxiv.org/abs/2407.16741). Номера разделов и таблиц
относятся к конференционному PDF. Современный OpenHands SDK здесь не рассматривается.

5 заданий. Под заголовком каждого указан его тип:

- **что есть что** — сопоставьте обезличенные компоненты с понятиями статьи;
- **расхождение со статьёй** — найдите одно отклонение от описанного устройства;
- **вопрос по коду** — код корректен, ответьте на вопросы через код и статью.

Для отклонения укажите место, объясните, что не так, приведите якорь в статье
и опишите исправление словами. Писать, запускать и отлаживать код не нужно.
Вопросы относятся только к заданиям без отклонений.

Общие импорты:

In [ ]:
import torch

Непоказанные вспомогательные функции считайте корректными. Это сокращённые модели
архитектуры, а не реализация API OpenHands. Сценарии независимы друг от друга.

## Задание 1. Три компонента

**Тип задания:** что есть что — код корректен, определите роли компонентов.

Сопоставьте `part_a`, `part_b`, `part_c` с **Agent**, **Runtime** и **State**.
`serialize` составляет вход модели из состояния; `parse` превращает ответ модели
в объект действия. `environment.execute` исполняет действие и возвращает наблюдение
(§2.1–2.2, рис. 2–3).

In [ ]:
def part_a(state, model, serialize, parse):
    messages = serialize(state)
    response = model(messages)
    return parse(response)

def part_b(action, environment):
    observation = environment.execute(action)
    return observation

def part_c(task, events, llm_cost):
    return {"task": task, "history": list(events), "llm_cost": llm_cost}

**Вопросы.**

1. Почему записи о выбранном действии недостаточно, чтобы узнать его результат?
   Какие события, помимо собственных действий агента, входят в историю (§2.1, рис. 2)?
2. Агенту добавили функцию `edit_file` из AgentSkills. Через какое уже существующее
   примитивное действие он может её вызвать? Как библиотека делает такие функции
   доступными разным агентам (§2.3)?

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 2. Две ячейки

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

Две последовательные Python-ячейки в одной задаче: сначала
`values = torch.tensor([1., 2., 3.])`, затем `total = values.sum()`.
`create_kernel()` создаёт новое пустое IPython-ядро в уже запущенном sandbox;
`kernel.execute` возвращает наблюдение, в том числе при ошибке исполнения.
Найдите одно отклонение в организации интерактивной Python-среды (§2.2, прил. F.1).

In [ ]:
def run_cells(cells, create_kernel):
    observations = []
    for cell in cells:
        kernel = create_kernel()
        observation = kernel.execute(cell)
        observations.append(observation)
    return observations

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 3. Команда из действия

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

Поступил `CmdRunAction`. `sandbox_api.run` исполняет команду через сервер API
в контейнере задачи, а `host_shell.run` — прямо на машине, где работает управляющий
процесс. Оба возвращают объект с `stdout` и `exit_code`. Найдите одно отклонение
от архитектуры исполнения (§2.2, прил. F.1, рис. 4).

In [ ]:
def execute_command(action, sandbox_api, host_shell):
    result = host_shell.run(action.command)
    return {"kind": "CmdRunObservation",
            "stdout": result.stdout,
            "exit_code": result.exit_code}

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 4. Образ для нового runtime

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

Подготовка runtime-образа. `content_tag(context)` зависит от Dockerfile и исходников
runtime API; `version_tag(base)` — общий тег для версии OpenHands и базового образа.
`build(parent, context)` собирает образ с текущим содержимым `context`.
Содержимое runtime API изменилось, а версия OpenHands и базовый образ остались прежними.
Найдите одно отклонение от алгоритма повторного использования образов
(прил. F.2.1–F.2.2, рис. 5).

In [ ]:
def prepare_image(base, context, cache, build, content_tag, version_tag):
    exact = content_tag(context)
    generic = version_tag(base)
    if cache.has(exact):
        return cache.get(exact)
    if cache.has(generic):
        return cache.get(generic)
    image = build(base, context)
    cache.tag(image, exact)
    cache.tag(image, generic)
    return image

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 5. Модель и цена ответа

**Тип задания:** вопрос по коду — в коде нет отклонений, ответьте на вопросы.

Три строки из табл. 7: CodeActAgent **v1.5** с GPT-3.5-turbo, GPT-4-turbo и GPT-4o.
Столбцы accuracy — GPQA Diamond / Main / Extended. Стоимость — средняя цена ответа
в долларах в экспериментах статьи (§4.4, прил. G).

In [ ]:
def gpqa_comparison():
    accuracy = torch.tensor([[27.9, 23.4, 26.1],
                             [51.8, 47.4, 42.4],
                             [53.1, 49.3, 52.8]])           # (3, 3), percent
    cost = torch.tensor([0.012, 0.501, 0.054])             # (3,), dollars
    gain = accuracy[2] - accuracy[1]                      # percentage points
    cost_ratio = cost[2] / cost[1]
    return gain, cost_ratio

**Вопросы.**

1. Как изменились accuracy на трёх поднаборах и стоимость при замене GPT-4-turbo
   на GPT-4o? Что здесь зафиксировано, а что меняется (табл. 7)?
2. Почему эти строки не измеряют отдельно эффект инструментов относительно обычного
   CoT? Опишите сравнение, которое отделит влияние модели от влияния агентной оболочки.
   Учитывайте версии CodeActAgent в табл. 3 и 7 (§4.1, §4.4, прил. G).

**Ваш ответ:**

_Напишите ответ здесь._